# Mapa de ataques hacker reales

**Proyecto final — DataXperience, Universidad EAN**
**Autor:** Sebastian Renteria Cruz

Un *honeypot* es un servidor expuesto a propósito en internet para registrar cada intento de ataque.
Este cuaderno analiza unos 450,000 ataques registrados por honeypots de AWS entre marzo y septiembre de 2013.

**Pregunta del proyecto:** ¿desde dónde, cuándo y contra qué servicios atacan los hackers,
y se puede predecir qué servicio van a atacar?

**Relación con la ciberseguridad:** el análisis de amenazas es la base del trabajo de un analista SOC.
Sirve para decidir qué puertos cerrar, qué servicios vigilar y a qué horas reforzar el monitoreo.

## Índice

1. Importaciones
2. Carga de datos
3. **Etapa 1 — Fundamentos y preparación de datos**
    1. Primera exploración
    2. Tipos de variables
    3. Detección de problemas
    4. Limpieza
4. **Etapa 2 — Análisis estadístico**
    1. Variables nuevas
    2. Estadística descriptiva
    3. ¿Desde dónde atacan?
    4. ¿Cuándo atacan?
    5. ¿Contra qué servicios?
    6. Valores atípicos (outliers)
    7. Hipótesis
5. **Etapa 3 — Modelado y storytelling**
    1. Modelo de clasificación: ¿qué servicio van a atacar?
    2. Regresión lineal: ¿los ataques van en aumento?
    3. Visualización final
    4. Storytelling: la historia de los datos
    5. Aplicación profesional
6. **Conclusiones**

## 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.compose import make_column_transformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, mean_absolute_error, r2_score)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Mostrar todas las columnas al imprimir tablas anchas
pd.set_option("display.max_columns", None)

# Estilo común para todas las gráficas
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Semilla única para todo lo aleatorio: los resultados se pueden reproducir
SEMILLA = 42

## 2. Carga de datos

El CSV se busca primero en la carpeta `datos/` del repositorio (ejecución local).
Si no está, como pasa en un Colab nuevo, se descarga desde la URL *raw* de GitHub.

In [ ]:
NOMBRE_CSV = "AWS_Honeypot_marx-geo.csv"

URL_RAW = f"https://raw.githubusercontent.com/RenteriaSc/mapa-ataques-honeypot/main/datos/{NOMBRE_CSV}"

# El cuaderno vive en notebook/, así que los datos están un nivel arriba
ruta_local = Path("..") / "datos" / NOMBRE_CSV
origen = ruta_local if ruta_local.exists() else URL_RAW

df_original = pd.read_csv(origen, low_memory=False)
print(f"Datos cargados desde: {origen}")

## 3. Etapa 1 — Fundamentos y preparación de datos

### 3.1 Primera exploración

Antes de cambiar nada, se mira el tamaño, los tipos de cada columna, unas filas de ejemplo
y las estadísticas básicas.

In [ ]:
filas, columnas = df_original.shape
print(f"Filas: {filas:,}  |  Columnas: {columnas}")

In [ ]:
df_original.info()

In [ ]:
df_original.head()

In [ ]:
df_original.describe()

**Interpretación:**

- El dataset tiene **451,581 ataques y 16 columnas**, entre el 3 de marzo y el 8 de septiembre de 2013.
- `datetime` se leyó como **texto**, no como fecha. Hay que convertirla para poder analizar horas y días.
- `type` solo tiene 44,811 valores, y `spt`/`dpt` tienen 406,770. Juntos suman exactamente las 451,581 filas.
  Esto **no es un error**: los ataques ICMP (como un *ping*) no usan puertos, sino un "tipo" de mensaje.
  Los ataques TCP y UDP sí usan puertos.
- `postalcode` solo está en el 19 % de las filas y `locale` en el 76 %. El país (`country`) está casi completo.
- La columna `Unnamed: 15` tiene apenas 83 valores, todos cercanos a -77.03, que es la longitud de Washington D.C.
  Y el máximo de `latitude` es **20,076**, imposible para una latitud (va de -90 a 90).
  Todo apunta a filas con las columnas corridas un lugar; se revisa en la sección 3.3.
- En `dpt` (puerto atacado), la mediana es **1433**, el puerto de Microsoft SQL Server.
  El 25 % de los ataques va a puertos de 445 hacia abajo, y la mitad al 1433 o menos. Las bases de datos y los servicios de Windows aparecen como objetivos principales.
- En `spt` (puerto de origen), el primer cuartil y la mediana valen **6000**. Un mismo puerto de origen repetido tantas veces
  suele ser la huella de una herramienta automática de escaneo, no de una persona.

### 3.2 Tipos de variables

Clasificar cada columna dice qué gráficas y qué tratamiento le corresponden:

| Tipo | Columnas | Ejemplo |
|---|---|---|
| Fecha | `datetime` | 3/3/13 21:53 |
| Categórica | `host`, `proto`, `cc`, `country`, `locale`, `localeabbr`, `postalcode` | groucho-oregon, TCP, CN |
| Numérica (identificador) | `spt`, `dpt` | puertos 6000, 1433 |
| Numérica (coordenada) | `latitude`, `longitude` | 28.55, 115.93 |
| Identificador de IP | `src` (IP como número), `srcstr` (IP como texto) | 61.131.218.218 |
| Por revisar | `type` y la columna extra sin nombre | — |

Los puertos son números, pero no se pueden promediar: el puerto 1433 no es "más" que el 22.
Son etiquetas de servicio, así que en el análisis se tratan como categorías.

In [ ]:
# Cuántos valores distintos tiene cada columna: ayuda a confirmar la tabla anterior
df_original.nunique().sort_values()

**Interpretación:**

- `proto` tiene solo **3 valores** (TCP, UDP e ICMP), y `host` tiene **9**: son los 9 honeypots de AWS repartidos por el mundo.
- `type` tiene 7 valores: son los códigos de mensaje ICMP. Por ejemplo, 8 es *echo request*, el típico *ping* de reconocimiento.
  Pasa de "por revisar" a **categórica**.
- `src` y `srcstr` tienen exactamente los mismos **69,602** valores distintos: son la misma IP escrita de dos formas.
  Para el análisis basta con una (`srcstr`).
- `cc` tiene 176 códigos de país y `country` tiene 177 nombres. Esa diferencia de uno se revisa en la sección 3.3.
- `dpt` tiene **4,041 puertos distintos**. Para el modelo habrá que agrupar los menos frecuentes, porque predecir entre 4,041 clases no es realista.
- `datetime` tiene 185,118 valores distintos en 451,581 filas: muchos ataques ocurren en el mismo minuto,
  algo que se espera de ataques automatizados.

### 3.3 Detección de problemas

Se revisan cinco tipos de problema: valores nulos, filas con columnas corridas, duplicados,
inconsistencias entre columnas y tipos de dato incorrectos. En esta sección **no se modifica nada**:
solo se mide. La corrección se hace en la sección 3.4.

#### Valores nulos

In [ ]:
def resumen_nulos(df):
    """Devuelve una tabla con la cantidad y el porcentaje de nulos por columna, de mayor a menor."""
    nulos = df.isna().sum()
    tabla = pd.DataFrame({
        "nulos": nulos,
        "porcentaje": (nulos / len(df) * 100).round(2),
    })
    return tabla[tabla["nulos"] > 0].sort_values("nulos", ascending=False)


resumen_nulos(df_original)

In [ ]:
# ¿Los nulos de type, spt y dpt dependen del protocolo?
pd.DataFrame({
    "filas": df_original["proto"].value_counts(),
    "con type": df_original.groupby("proto")["type"].count(),
    "con dpt": df_original.groupby("proto")["dpt"].count(),
})

**Interpretación:** hay dos clases de nulos.

- **Nulos estructurales (no son errores):** cada fila ICMP tiene `type` y ninguna tiene puertos.
  Con TCP y UDP pasa justo lo contrario. Un *ping* no va dirigido a un puerto, así que esos valores vacíos son correctos y **se conservan**.
- **Nulos reales:** faltan `postalcode` (81 %), `localeabbr` y `locale` (24-27 %) y las coordenadas de unas 3,400 filas.
  La base de geolocalización no siempre logra ubicar una IP con tanta precisión. Estas filas **también se conservan**,
  porque siguen siendo útiles para analizar horarios y servicios. Solo se dejan fuera del mapa.
- `Unnamed: 15` es casi toda nula: es una columna que no debería existir (ver el siguiente punto).

#### Filas con las columnas corridas

In [ ]:
filas_corridas = df_original["Unnamed: 15"].notna()
print(f"Filas con valor en 'Unnamed: 15': {filas_corridas.sum()}")
print(f"Latitudes fuera del rango [-90, 90]: {(df_original['latitude'].abs() > 90).sum()}")

df_original.loc[filas_corridas, ["locale", "localeabbr", "postalcode", "latitude", "longitude", "Unnamed: 15"]] \
    .value_counts(dropna=False).head()

**Interpretación:** las 83 filas son de **Washington, D.C.** El nombre del lugar trae una coma
y en el CSV no viene entre comillas, así que pandas lo partió en dos columnas (`Washington` y `D.C.`).
Todo lo que viene después quedó corrido un lugar a la derecha:

| Columna donde quedó | Valor que en realidad es |
|---|---|
| `localeabbr` = "D.C." | parte de `locale` |
| `postalcode` = "DC" | `localeabbr` |
| `latitude` (ej. 20057) | `postalcode` |
| `longitude` (ej. 38.89) | `latitude` |
| `Unnamed: 15` (ej. -77.03) | `longitude` |

Por eso aparecen latitudes imposibles como 20,076: en realidad son códigos postales.
Los datos no están perdidos, solo desordenados, así que **se realinean en vez de borrarse**.

#### Filas duplicadas

In [ ]:
duplicadas = df_original.duplicated()
print(f"Filas idénticas a otra anterior: {duplicadas.sum():,} ({duplicadas.mean() * 100:.2f} %)")

# Ejemplo: el grupo de filas idénticas más grande
columnas = list(df_original.columns)
df_original[df_original.duplicated(keep=False)] \
    .groupby(columnas, dropna=False).size().sort_values(ascending=False).head(3)

**Interpretación:** casi una de cada cuatro filas es idéntica a otra. Pero `datetime` solo registra
hasta el **minuto**. Si una herramienta automática envía varios intentos desde la misma IP y el mismo puerto
al mismo servicio en un minuto, todos quedan registrados igual, aunque cada uno sea un intento distinto.
La tabla lo muestra: el grupo más grande son **2,310 filas idénticas** de una misma IP de Irán contra el puerto 2193
del honeypot de Tokyo, todas en el minuto 7:47 del 24 de julio. Ninguna persona hace eso a mano: es una ráfaga automatizada.

**Decisión:** los duplicados **se conservan**. Borrarlos haría creer que hubo menos ataques de los que hubo
y ocultaría justo el comportamiento más agresivo, la ráfaga automatizada. Para un analista de seguridad,
el volumen de intentos es información en sí misma.

#### Inconsistencias entre país y código de país

In [ ]:
# Filas que tienen nombre de país pero no código, y al revés
print("Con country pero sin cc:")
print(df_original.loc[df_original["cc"].isna() & df_original["country"].notna(), ["cc", "country"]])

sin_nombre = df_original["cc"].notna() & df_original["country"].isna()
print(f"\nCon cc pero sin country: {sin_nombre.sum()}")
df_original.loc[sin_nombre, "cc"].value_counts()

**Interpretación:**

- La única fila de **Namibia** se quedó sin código de país porque su código ISO es `NA`, y pandas
  lee el texto "NA" como "valor faltante". Es un error típico de pandas: el dato estaba bien en el archivo.
  Se corrige poniendo `cc = "NA"` en esa fila.
- 39 filas tienen código de país pero no nombre. En otras filas el mismo código sí trae nombre
  (por ejemplo, `US` siempre es *United States*), así que el nombre **se completa** con la correspondencia
  código → país del propio dataset. No se inventa nada.

#### Tipos de dato y columnas redundantes

In [ ]:
# src y srcstr: ¿cada número corresponde siempre a una sola IP en texto?
print("IP en texto por cada src (máximo):", df_original.groupby("src")["srcstr"].nunique().max())

# ¿Todas las fechas tienen el formato mes/día/año hora:minuto?
fechas = pd.to_datetime(df_original["datetime"], format="%m/%d/%y %H:%M", errors="coerce")
print("Fechas que no se pudieron convertir:", fechas.isna().sum())
print("Rango:", fechas.min(), "→", fechas.max())

**Interpretación:**

- `src` es la misma IP de `srcstr`, solo que escrita como número (relación 1 a 1). Se **elimina** `src` por redundante.
- Las 451,581 fechas siguen el formato `mes/día/año hora:minuto` y se convierten sin errores a tipo fecha.
  Eso permite extraer la hora, el día de la semana y el mes en la Etapa 2.
- `spt`, `dpt` y `type` se leyeron como decimales (`6000.0`) por culpa de los nulos. Se pasan a enteros
  que admiten nulos (`Int64`), porque un puerto 1433.5 no existe.

#### Resumen de problemas y decisiones

| Problema | Filas | Decisión |
|---|---|---|
| Nulos de `type` / `spt` / `dpt` según protocolo | 451,581 | Conservar: son correctos |
| Nulos de ubicación (`postalcode`, `locale`, coordenadas) | variable | Conservar; fuera del mapa si no hay coordenadas |
| Columnas corridas (Washington, D.C.) | 83 | Realinear |
| Duplicados exactos | 108,030 | Conservar: son intentos reales |
| Namibia leída como nulo | 1 | `cc = "NA"` |
| País sin nombre | 39 | Completar desde el código |
| `datetime` como texto | 451,581 | Convertir a fecha |
| `src` redundante, `Unnamed: 15` sobrante | — | Eliminar columnas |
| Puertos y `type` como decimales | — | Convertir a `Int64` |

### 3.4 Limpieza

Toda la limpieza vive en una sola función. Así se puede leer de principio a fin y volver a aplicar
sobre el CSV original. El original **no se modifica**: el resultado se guarda aparte en `datos/ataques_limpio.csv`.

In [ ]:
def realinear_washington(df):
    """Corrige las filas de 'Washington, D.C.' cuyas columnas quedaron corridas un lugar a la derecha."""
    df = df.copy()
    corridas = df["Unnamed: 15"].notna()
    # Se lee todo antes de escribir, para no pisar un valor que todavía se necesita
    codigo_postal = df.loc[corridas, "latitude"]
    latitud = df.loc[corridas, "longitude"]
    longitud = df.loc[corridas, "Unnamed: 15"]

    df.loc[corridas, "locale"] = "Washington, D.C."
    df.loc[corridas, "localeabbr"] = "DC"
    df.loc[corridas, "postalcode"] = codigo_postal.map(lambda v: f"{v:.0f}", na_action="ignore")
    df.loc[corridas, "latitude"] = latitud
    df.loc[corridas, "longitude"] = longitud
    return df


def limpiar_ataques(df):
    """Aplica todas las decisiones de la sección 3.3 y devuelve un DataFrame limpio nuevo."""
    df = realinear_washington(df)
    df = df.drop(columns=["Unnamed: 15", "src"])

    # Namibia: su código "NA" fue leído como nulo
    df.loc[df["cc"].isna() & (df["country"] == "Namibia"), "cc"] = "NA"

    # Completar el nombre del país usando la correspondencia código → país del propio dataset
    codigo_a_pais = df.dropna(subset=["cc", "country"]).drop_duplicates("cc").set_index("cc")["country"]
    df["country"] = df["country"].fillna(df["cc"].map(codigo_a_pais))

    df["datetime"] = pd.to_datetime(df["datetime"], format="%m/%d/%y %H:%M")
    df[["spt", "dpt", "type"]] = df[["spt", "dpt", "type"]].astype("Int64")
    return df


df = limpiar_ataques(df_original)
df.info()

In [ ]:
# Comprobaciones: si alguna falla, el cuaderno se detiene aquí con un mensaje claro
assert len(df) == len(df_original), "La limpieza no debe borrar filas"
assert df["latitude"].dropna().between(-90, 90).all(), "Hay latitudes fuera de rango"
assert df["longitude"].dropna().between(-180, 180).all(), "Hay longitudes fuera de rango"
assert (df["cc"].isna() == df["country"].isna()).all(), "cc y country deben estar completos o vacíos a la vez"
print("Todas las comprobaciones pasaron ✔")

resumen_nulos(df)

**Interpretación:** el dataset limpio conserva las **451,581 filas** y queda con 14 columnas.
Todas las coordenadas están ahora en rangos válidos, cada país tiene su código y su nombre,
y `datetime` ya es una fecha. Los nulos que quedan son los esperados: los estructurales (ICMP sin puertos,
TCP/UDP sin `type`) y los de ubicación que la geolocalización no pudo resolver.

In [ ]:
# Fuera de Colab se guarda en datos/; en un Colab nuevo, junto al cuaderno
carpeta_datos = Path("..") / "datos"
ruta_limpio = (carpeta_datos if carpeta_datos.exists() else Path(".")) / "ataques_limpio.csv"

df.to_csv(ruta_limpio, index=False)
print(f"Datos limpios guardados en: {ruta_limpio}  ({ruta_limpio.stat().st_size / 1e6:.1f} MB)")

## 4. Etapa 2 — Análisis estadístico

**El reto de este dataset:** casi ninguna columna es un número que se pueda promediar. Un puerto es una etiqueta
(el 1433 no es "más" que el 22) y las coordenadas son posiciones. Las medidas estadísticas se calculan entonces sobre
**conteos**: cuántos ataques hay por día, por IP atacante y por hora.

> **Nota sobre la hora:** el dataset no documenta la zona horaria de `datetime`. Las horas se muestran como "hora del registro".
> Más abajo se ve que esta limitación no cambia la conclusión principal sobre el horario.

### 4.1 Variables nuevas

A partir de la fecha se obtienen la hora, el día de la semana y el día del calendario.
A partir del puerto se obtiene el nombre del servicio, que es mucho más fácil de leer para un analista.

In [ ]:
# Nombre de los puertos más atacados (puertos estándar de cada servicio)
SERVICIOS = {
    1433: "SQL Server (base de datos)",
    445: "SMB (archivos de Windows)",
    3389: "RDP (escritorio remoto)",
    80: "HTTP (web)",
    8080: "HTTP alternativo / proxy",
    22: "SSH (acceso remoto)",
    3306: "MySQL (base de datos)",
    135: "RPC de Windows",
    23: "Telnet (acceso remoto)",
    25: "SMTP (correo)",
    53: "DNS (nombres de dominio)",
}
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

df["fecha"] = df["datetime"].dt.date
df["hora"] = df["datetime"].dt.hour
df["dia_semana"] = pd.Categorical(df["datetime"].dt.day_of_week.map(dict(enumerate(DIAS))), categories=DIAS, ordered=True)
df["servicio"] = df["dpt"].map(SERVICIOS)
# Sin nombre conocido se deja el número; ICMP no usa puertos
df["servicio"] = df["servicio"].fillna("Puerto " + df["dpt"].astype("string")).fillna("ICMP (sin puerto)")

df[["datetime", "fecha", "hora", "dia_semana", "dpt", "servicio"]].head()

### 4.2 Estadística descriptiva

Se arman tres variables numéricas contando ataques:

- **Ataques por día:** 190 días con registro.
- **Ataques por IP:** cuántos ataques hizo cada una de las 69,602 IPs atacantes.
- **Ataques por hora del día:** total acumulado de cada una de las 24 horas.

In [ ]:
def tabla_estadisticas(variables):
    """Recibe un diccionario {nombre: Series} y devuelve una tabla con las medidas de tendencia central y dispersión."""
    filas = {}
    for nombre, serie in variables.items():
        filas[nombre] = {
            "n": len(serie),
            "media": serie.mean(),
            "mediana": serie.median(),
            "moda": serie.mode().iloc[0],
            "mínimo": serie.min(),
            "máximo": serie.max(),
            "rango": serie.max() - serie.min(),
            "varianza": serie.var(),
            "desv. estándar": serie.std(),
            "coef. variación (%)": serie.std() / serie.mean() * 100,
            "asimetría": serie.skew(),
        }
    return pd.DataFrame(filas).T.round(2)


ataques_por_dia = df.groupby("fecha").size()
ataques_por_ip = df.groupby("srcstr").size()
ataques_por_hora = df.groupby("hora").size()

tabla_estadisticas({
    "Ataques por día": ataques_por_dia,
    "Ataques por IP": ataques_por_ip,
    "Ataques por hora del día": ataques_por_hora,
})

**Interpretación:**

- **Ataques por día:** la media (2,376.74) es mayor que la mediana (2,204.50), y la asimetría es muy positiva (6.77).
  Unos pocos días con muchísimos ataques (el máximo es 13,474) empujan la media hacia arriba.
  **La mediana representa mejor un día típico: unos 2,200 ataques**, es decir, más de 90 por hora contra solo 9 servidores.
- **Ataques por IP:** este es el caso más extremo. La mediana y la moda son **2 ataques**, pero la media es 6.49
  y el máximo es **18,472**. La desviación estándar (102.99) es casi 16 veces la media. Casi todos los atacantes
  hacen 1 o 2 intentos, y unos pocos hacen miles. Aquí la media no sirve para describir a un atacante típico.
- **Ataques por hora:** la dispersión es baja (coeficiente de variación del 18.86 %). Más abajo se ve que esa poca
  variación se debe casi toda a dos atacantes puntuales.

**Para un analista:** el atacante típico prueba un par de veces y se va. El peligro real está en los pocos que insisten miles de veces.

### 4.3 ¿Desde dónde atacan?

El mapa agrupa los ataques por coordenadas. Cada burbuja es un lugar, y su tamaño indica la cantidad de ataques.
Es interactivo: se puede acercar y pasar el mouse por encima. Las ~3,400 filas sin coordenadas no aparecen.

In [ ]:
por_lugar = (
    df.dropna(subset=["latitude", "longitude"])
    .groupby(["latitude", "longitude", "country"], as_index=False)
    .size()
    .rename(columns={"size": "ataques"})
)

fig = px.scatter_geo(
    por_lugar, lat="latitude", lon="longitude", size="ataques", color="country",
    hover_name="country", hover_data={"ataques": ":,", "latitude": False, "longitude": False},
    projection="natural earth", size_max=40,
    title="Mapa de ataques a los honeypots de AWS (marzo–septiembre 2013)",
)
fig.update_layout(showlegend=False, height=550)
fig.show()

In [ ]:
top_paises = df["country"].value_counts().head(10)
porcentaje = (top_paises / len(df) * 100).round(1)

ax = top_paises.sort_values().plot.barh(color="steelblue")
ax.set_title("Top 10 países de origen de los ataques")
ax.set_xlabel("Número de ataques")
ax.set_ylabel("País")
for i, (valor, pct) in enumerate(zip(top_paises.sort_values(), porcentaje.sort_values())):
    ax.text(valor, i, f" {pct} %", va="center")
plt.show()

print(f"Países distintos: {df['country'].nunique()}")
print(f"China + Estados Unidos: {porcentaje.iloc[:2].sum():.1f} % de todos los ataques")

In [ ]:
# ¿Y desde Colombia?
colombia = df[df["country"] == "Colombia"]
print(f"Ataques desde Colombia: {len(colombia):,} ({len(colombia) / len(df) * 100:.2f} %)")
colombia["servicio"].value_counts().head(5)

**Interpretación:**

- **China (42.4 %) y Estados Unidos (19.9 %) originan el 62.3 % de los ataques**, casi dos de cada tres. Después hay un salto enorme:
  el tercero, Japón, tiene apenas el 3.8 %.
- Que una IP sea de un país **no significa que el atacante viva allí**. Muchos ataques salen de equipos infectados
  o de servidores alquilados. Lo que sí muestra el mapa es dónde están las máquinas desde las que se ataca.
- Desde Colombia hubo **582 ataques (0.13 %)**, sobre todo contra SMB (archivos de Windows).

**Para un analista:** el bloqueo por país es una medida tentadora, pero solo con China y Estados Unidos se dejaría
por fuera a una parte enorme del tráfico legítimo. Es mejor filtrar por servicio y por comportamiento.

### 4.4 ¿Cuándo atacan?

In [ ]:
ax = ataques_por_hora.plot.bar(color="steelblue")
ax.set_title("Ataques por hora del día")
ax.set_xlabel("Hora del registro (0–23)")
ax.set_ylabel("Número de ataques")
ax.axhline(ataques_por_hora.median(), color="red", linestyle="--", label=f"Mediana ({ataques_por_hora.median():,.0f})")
ax.legend()
plt.xticks(rotation=0)
plt.show()

In [ ]:
# ¿Quién causa los picos de las horas 0, 7 y 23?
for hora in [0, 7, 23]:
    ip, cantidad = df.loc[df["hora"] == hora, "srcstr"].value_counts().head(1).items().__next__()
    pais = df.loc[df["srcstr"] == ip, "country"].iloc[0]
    print(f"Hora {hora:>2}: la IP {ip} ({pais}) aporta {cantidad:,} de {ataques_por_hora[hora]:,} ataques")

# Sin los 5 atacantes más activos, ¿cómo queda el horario?
top5_ips = ataques_por_ip.nlargest(5).index
sin_top5 = df[~df["srcstr"].isin(top5_ips)].groupby("hora").size()
print(f"\nCoeficiente de variación por hora con todos: {ataques_por_hora.std() / ataques_por_hora.mean() * 100:.1f} %")
print(f"Coeficiente de variación por hora sin el top 5: {sin_top5.std() / sin_top5.mean() * 100:.1f} %")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

df["dia_semana"].value_counts().sort_index().plot.bar(ax=ejes[0], color="steelblue")
ejes[0].set_title("Ataques por día de la semana")
ejes[0].set_xlabel("Día de la semana")
ejes[0].set_ylabel("Número de ataques")
ejes[0].tick_params(axis="x", rotation=45)

ataques_por_dia.plot(ax=ejes[1], color="steelblue")
ejes[1].set_title("Ataques por día (marzo–septiembre 2013)")
ejes[1].set_xlabel("Fecha")
ejes[1].set_ylabel("Número de ataques")

plt.tight_layout()
plt.show()

**Interpretación:**

- Los picos de las horas 0, 7 y 23 los causan **dos IPs**: una de Irán (11,116 ataques en una sola hora del 24 de julio)
  y una de China (18,195 ataques entre la noche del 26 y la del 27 de agosto).
- **Sin los 5 atacantes más activos, la variación entre horas baja a cerca del 7.5 %**: se ataca casi igual a las 3 a. m.
  que a las 3 p. m. Una persona trabaja con horario; un programa automático no. Por eso no conocer la zona horaria
  no cambia la conclusión.
- Entre semana, de lunes a miércoles, hay algo más de ataques (~70 mil por día) que de jueves a domingo (~60 mil).
- La gráfica por día muestra pocos picos aislados sobre una línea bastante estable. Esos picos se analizan en la sección 4.6.

**Para un analista:** el monitoreo debe ser 24/7. No existe una "hora segura".

### 4.5 ¿Contra qué servicios atacan?

In [ ]:
top_servicios = df["servicio"].value_counts().head(10)

ax = top_servicios.sort_values().plot.barh(color="darkorange")
ax.set_title("Top 10 servicios atacados")
ax.set_xlabel("Número de ataques")
ax.set_ylabel("Servicio (puerto de destino)")
plt.show()

print(f"Protocolos:\n{df['proto'].value_counts(normalize=True).mul(100).round(1).to_string()}")

In [ ]:
# Cruce: ¿cada país prefiere servicios distintos? (porcentaje de los ataques de cada país)
paises = df["country"].value_counts().head(8).index
servicios = df["servicio"].value_counts().head(8).index
cruce = pd.crosstab(df["country"], df["servicio"], normalize="index").loc[paises, servicios] * 100

ax = sns.heatmap(cruce, annot=True, fmt=".0f", cmap="Oranges", cbar_kws={"label": "% de los ataques del país"})
ax.set_title("Servicio atacado según país de origen (%)")
ax.set_xlabel("Servicio")
ax.set_ylabel("País de origen")
plt.xticks(rotation=40, ha="right")
plt.show()

In [ ]:
# Servicio favorito de cada país, aunque no esté entre los 8 más atacados del mapa de calor
favorito = (
    df[df["country"].isin(paises)]
    .groupby("country")["servicio"]
    .agg(servicio_favorito=lambda s: s.value_counts().index[0],
         porcentaje=lambda s: s.value_counts(normalize=True).iloc[0] * 100)
    .loc[paises]
    .round(1)
)
favorito

**Interpretación:**

- **SQL Server (1433) es el blanco número uno, con 109,425 ataques**: casi uno de cada cuatro. El segundo lugar son los pings
  **ICMP** (44,811): no atacan un servicio, solo comprueban si el servidor existe. Es la fase de **reconocimiento**.
  Después vienen SMB (445) y el escritorio remoto RDP (3389). Los atacantes buscan **bases de datos y equipos Windows** mal protegidos.
- Los puertos 56338 y 2193 no son servicios estándar. Casi todo su volumen viene de las dos IPs de las ráfagas
  de China e Irán vistas antes.
- El mapa de calor y la tabla del servicio favorito muestran que **cada país tiene un "perfil" distinto**:
  China apunta sobre todo a SQL Server (44 %), Japón casi solo hace pings ICMP (52 %), Taiwán prefiere SMB (25 %)
  y los ataques de Estados Unidos están más repartidos. Irán y Países Bajos salen casi en blanco en el mapa de calor
  porque su blanco favorito no está entre los 8 servicios más comunes: Irán dirige el 85 % al puerto 2193
  (la ráfaga del 24 de julio) y Países Bajos el 37 % a DNS.
  Si el país ayuda a adivinar el servicio, es una buena pista para el modelo de la Etapa 3.

**Para un analista:** SQL Server, SMB y RDP nunca deberían quedar expuestos directamente a internet.

### 4.6 Valores atípicos (outliers)

Se usa la **regla IQR**: un valor es atípico si está por debajo de Q1 − 1.5·IQR o por encima de Q3 + 1.5·IQR,
donde IQR = Q3 − Q1 (el rango del 50 % central de los datos).

In [ ]:
def outliers_iqr(serie):
    """Devuelve los límites de la regla IQR y los valores de la serie que quedan por fuera."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    atipicos = serie[(serie < limite_inferior) | (serie > limite_superior)]
    return limite_inferior, limite_superior, atipicos


resumen = {}
for nombre, serie in {"Ataques por día": ataques_por_dia, "Ataques por IP": ataques_por_ip}.items():
    inf, sup, atipicos = outliers_iqr(serie)
    resumen[nombre] = {
        "límite inferior": inf,
        "límite superior": sup,
        "cantidad de atípicos": len(atipicos),
        "% de la variable": len(atipicos) / len(serie) * 100,
        "% de todos los ataques": atipicos.sum() / serie.sum() * 100,
    }
pd.DataFrame(resumen).T.round(2)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=ataques_por_dia, ax=ejes[0], color="steelblue")
ejes[0].set_title("Ataques por día")
ejes[0].set_xlabel("Ataques en un día")

sns.boxplot(x=ataques_por_ip, ax=ejes[1], color="darkorange")
# El eje se pasa a logarítmico DESPUÉS de dibujar: así la caja y los bigotes siguen la regla IQR
# sobre los valores reales (límite 8.5), y la caja de 1 a 4 ataques no queda aplastada contra el 0
ejes[1].set_xscale("log")
ejes[1].set_title("Ataques por IP (escala logarítmica)")
ejes[1].set_xlabel("Ataques hechos por una IP")

plt.tight_layout()
plt.show()

In [ ]:
# ¿Quiénes son los días atípicos?
_, _, dias_atipicos = outliers_iqr(ataques_por_dia)
dias_atipicos.sort_values(ascending=False)

**Interpretación:** ¿son errores o ataques reales?

- **Días atípicos por arriba:** son días de ráfaga. El 24 de julio (13,474) coincide con la IP de Irán, y el 26 y el 27 de agosto
  (más de 11 mil cada uno) con la IP de China. Son **ataques reales**, no errores, y **se conservan**: son justo lo que un SOC debe detectar.
- **Días atípicos por abajo:** el 3 de marzo y el 8 de septiembre son el **primer y el último día** del registro.
  Empezó a las 21:53 y terminó a las 5:55, así que son días incompletos, no días tranquilos.
- **IPs atípicas:** el límite es de solo 8.5 ataques. Las IPs que lo superan son menos del 10 % de los atacantes,
  pero hicieron **dos de cada tres ataques**. Es el principio de Pareto aplicado a la seguridad: pocos atacantes,
  mucho volumen.

**Para un analista:** una regla simple como "alertar si una IP supera ~9 intentos" ya separaría al atacante ocasional
del persistente.

### 4.7 Hipótesis

Cada hipótesis sale de un valor concreto de las tablas anteriores:

| # | Hipótesis | Evidencia |
|---|---|---|
| H1 | **El servicio atacado depende del país de origen.** | China dirige el 44 % de sus ataques a SQL Server, Japón el 52 % a ICMP y Taiwán el 25 % a SMB. |
| H2 | **Los ataques son automatizados y no siguen un horario humano.** | Sin el top 5 de IPs, la variación entre horas es de apenas ~7.5 %. |
| H3 | **Unos pocos atacantes generan la mayoría del volumen.** | Las IPs atípicas (menos del 10 %) hacen cerca del 66 % de los ataques. |
| H4 | **La ubicación del honeypot influye en cuánto lo atacan.** | Los 9 estuvieron activos los mismos 188 días, pero Tokyo recibió 126,189 ataques y Europa solo 23,954. |

H1 y H4 sugieren que el **país de origen** y el **honeypot atacado** son buenas variables para predecir el servicio
en la Etapa 3. H2 sugiere que la **hora**, en cambio, aportará poco.

## 5. Etapa 3 — Modelado y storytelling

### 5.1 Modelo de clasificación: ¿qué servicio van a atacar?

**¿Por qué no una regresión lineal aquí?** La regresión lineal predice **números**, como "cuántos ataques habrá mañana".
La pregunta del proyecto es **qué servicio** van a atacar, y la respuesta es una **categoría** (SQL Server, SMB, RDP…).
Para eso se usa un **modelo de clasificación**. La regresión lineal se usa más abajo, en la sección 5.2, donde sí hay un número que predecir.

**¿Por qué un árbol de decisión?** Funciona como el juego de "adivina quién": hace preguntas de sí o no
(*¿viene de China? ¿usa el puerto de origen 6000?*) hasta llegar a una respuesta. Se puede dibujar
y explicar sin fórmulas, algo clave para que un equipo de seguridad confíe en él.

**Cómo se plantea:**

- **Qué se predice:** el servicio atacado, con los 5 servicios estándar más frecuentes por nombre y el resto como "Otro".
  Se dejan fuera los pings ICMP porque no atacan ningún puerto.
- **Con qué se predice:** país de origen, honeypot atacado, protocolo, si el puerto de origen es 6000
  (la huella de escáner vista en la sección 3.1), hora y día de la semana.
- **Qué NO se usa:** el puerto de destino (`dpt`). Es justo la respuesta, así que usarlo sería hacer trampa (*fuga de datos*).

In [ ]:
modelo_df = df[df["proto"] != "ICMP"].copy()

# Variable objetivo: los 5 servicios estándar más atacados y el resto agrupado en "Otro".
# Los puertos no estándar (como el 56338) se van a "Otro": casi todo su volumen es UN solo atacante,
# y un modelo no puede aprender a reconocer una clase que es una sola IP
con_nombre = modelo_df["dpt"].isin(SERVICIOS.keys())
top_servicios_modelo = modelo_df.loc[con_nombre, "servicio"].value_counts().head(5).index
modelo_df["objetivo"] = modelo_df["servicio"].where(modelo_df["servicio"].isin(top_servicios_modelo), "Otro")

# Los países poco frecuentes se agrupan para que el árbol no aprenda casos de un solo atacante
top_paises_modelo = modelo_df["country"].value_counts().head(15).index
modelo_df["pais"] = modelo_df["country"].where(modelo_df["country"].isin(top_paises_modelo), "Otro").fillna("Otro")
modelo_df["origen_6000"] = (modelo_df["spt"] == 6000).map({True: "sí", False: "no"})

VARIABLES = ["pais", "host", "proto", "origen_6000", "hora", "dia_semana"]
X = modelo_df[VARIABLES].astype(str)
y = modelo_df["objetivo"]
grupos = modelo_df["srcstr"]

y.value_counts(normalize=True).mul(100).round(1).rename("% de los ataques").to_frame()

#### Separar entrenamiento y prueba: una trampa que hay que evitar

En la sección 3.3 se vio que hay muchas filas idénticas (las ráfagas). Si se separan los datos **al azar**,
copias exactas del mismo ataque quedan en entrenamiento y en prueba. El modelo "memoriza" y parece mejor de lo que es.

La solución es **separar por IP**: todos los ataques de una misma IP van al mismo lado.
Así el modelo se evalúa con **atacantes que nunca vio**, igual que pasaría en la vida real.

In [ ]:
separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMILLA)
idx_entrena, idx_prueba = next(separador.split(X, y, groups=grupos))

X_entrena, X_prueba = X.iloc[idx_entrena], X.iloc[idx_prueba]
y_entrena, y_prueba = y.iloc[idx_entrena], y.iloc[idx_prueba]

ips_compartidas = set(grupos.iloc[idx_entrena]) & set(grupos.iloc[idx_prueba])
print(f"Entrenamiento: {len(X_entrena):,} ataques  |  Prueba: {len(X_prueba):,} ataques")
print(f"IPs presentes en ambos conjuntos: {len(ips_compartidas)}")

#### Elegir la profundidad del árbol

Un árbol muy profundo memoriza (*sobreajuste*) y uno muy corto se queda simple. La profundidad se elige con
**validación cruzada** (5 rondas, también separadas por IP) y **solo con los datos de entrenamiento**.
Los datos de prueba no se tocan hasta el final.

In [ ]:
def crear_arbol(profundidad):
    """Arma el flujo completo: convierte las categorías en columnas 0/1 y entrena un árbol de decisión."""
    codificador = make_column_transformer((OneHotEncoder(handle_unknown="ignore"), VARIABLES))
    return make_pipeline(codificador, DecisionTreeClassifier(max_depth=profundidad, random_state=SEMILLA))


resultados_cv = {}
for profundidad in [2, 4, 6, 8, 10, 12]:
    puntajes = cross_val_score(crear_arbol(profundidad), X_entrena, y_entrena,
                               groups=grupos.iloc[idx_entrena], cv=GroupKFold(n_splits=5), scoring="accuracy")
    resultados_cv[profundidad] = {"accuracy promedio": puntajes.mean(), "desviación": puntajes.std()}

tabla_cv = pd.DataFrame(resultados_cv).T.round(3)
tabla_cv.index.name = "profundidad"
mejor_profundidad = int(tabla_cv["accuracy promedio"].idxmax())
print(f"Mejor profundidad: {mejor_profundidad}")
tabla_cv

In [ ]:
modelo_base = DummyClassifier(strategy="most_frequent").fit(X_entrena, y_entrena)
arbol = crear_arbol(mejor_profundidad).fit(X_entrena, y_entrena)

pred_base = modelo_base.predict(X_prueba)
pred_arbol = arbol.predict(X_prueba)

pd.DataFrame({
    "accuracy": [accuracy_score(y_prueba, pred_base), accuracy_score(y_prueba, pred_arbol)],
    "F1 promedio (macro)": [f1_score(y_prueba, pred_base, average="macro"), f1_score(y_prueba, pred_arbol, average="macro")],
}, index=[f"Modelo base (siempre '{modelo_base.classes_[modelo_base.class_prior_.argmax()]}')", "Árbol de decisión"]).round(3)

**Interpretación:**

- El **modelo base** no aprende nada: siempre responde "Otro", la clase más común, y acierta el **45.2 %** de las veces.
  Es el mínimo que cualquier modelo debe superar.
- El **árbol de decisión acierta el 65.7 %**: **20.5 puntos más** que el modelo base, con atacantes que nunca vio.
- El F1 promedio sube de 0.10 a 0.41. El F1 mide qué tan bien se reconoce *cada* servicio, no solo el más común.
  Este valor más modesto indica que el árbol reconoce muy bien unos servicios y mal otros (ver la tabla siguiente).
- En la validación cruzada el árbol acertó un 68.4 %, muy cerca del 65.7 % de la prueba. Esto indica que **no está memorizando**:
  se comporta parecido con datos nuevos.

In [ ]:
# F1 por clase: qué tan bien reconoce cada servicio
reporte = pd.DataFrame(classification_report(y_prueba, pred_arbol, output_dict=True, zero_division=0)).T
reporte = reporte.drop(index=["accuracy", "macro avg", "weighted avg"])
reporte.columns = ["precisión", "recall", "F1", "ataques en prueba"]
reporte.round(2).sort_values("F1", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_arbol, normalize="true", values_format=".0%",
                                        cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Matriz de confusión del árbol (% de cada servicio real)")
ax.set_xlabel("Servicio que predijo el modelo")
ax.set_ylabel("Servicio realmente atacado")
plt.xticks(rotation=40, ha="right")
plt.show()

**Interpretación:** cómo leer estas métricas:

- **Precisión:** de las veces que el modelo dijo "SQL Server", cuántas acertó.
- **Recall:** de todos los ataques reales a SQL Server, cuántos detectó.
- **F1:** un promedio de las dos. 1 es perfecto y 0 es nunca acertar.

Resultados:

- **SQL Server** es el que mejor reconoce: detecta el **88 %** de sus ataques (F1 = 0.72).
  **SMB** también va bien: detecta el 69 % (F1 = 0.71).
- **HTTP** solo se detecta en el 21 % de los casos. El 64 % se confunde con "Otro".
- **RDP y HTTP alternativo nunca se predicen.** La matriz muestra por qué: los ataques a RDP se reparten casi
  por igual entre "SQL Server" (48 %) y "Otro" (49 %). Con las variables disponibles (país, honeypot, protocolo, puerto de origen),
  un ataque a RDP **se ve igual** que uno a SQL Server. Para separarlos haría falta más información, por ejemplo el historial de la IP.

In [ ]:
# ¿Qué variables usa más el árbol? Se suman las columnas 0/1 de cada variable original
nombres = arbol[0].get_feature_names_out()
importancia = pd.Series(arbol[-1].feature_importances_, index=nombres)
importancia = importancia.groupby(lambda n: n.split("__")[1].rsplit("_", 1)[0] if "__" in n else n).sum()
importancia = importancia.reindex(VARIABLES).sort_values() * 100

ax = importancia.plot.barh(color="seagreen")
ax.set_title("Importancia de cada variable en el árbol")
ax.set_xlabel("Importancia (%)")
ax.set_ylabel("Variable")
plt.show()

In [ ]:
# Las primeras 3 preguntas del árbol (el árbol completo tiene más niveles y no se leería)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(arbol[-1], max_depth=2, feature_names=[n.split("__")[1] for n in nombres],
          class_names=arbol[-1].classes_, filled=True, impurity=False, proportion=True, fontsize=9, ax=ax)
ax.set_title("Primeras preguntas del árbol de decisión")
plt.show()

**Interpretación:**

- **La pregunta más útil es "¿el puerto de origen es 6000?"**, con el 43 % de la importancia. Es la primera pregunta del árbol.
  Cuando la respuesta es *sí*, el 62 % de esos ataques van contra SQL Server, y si además el blanco es el honeypot de Oregon,
  la proporción sube al 84 %. El puerto 6000 es la huella de una herramienta automática que escanea bases de datos.
- Después pesan el **honeypot atacado** (24 %), el **protocolo** (20 %) y el **país** (13 %).
- **La hora y el día de la semana no aportan casi nada** (menos del 1 %). Esto **confirma la hipótesis H2**: los ataques son
  automáticos y no siguen un horario. **H4** (la ubicación del honeypot importa) también se confirma.
  **H1** (el país importa) se confirma solo en parte: el país ayuda, pero menos que el puerto de origen y el honeypot.

#### Prueba de la trampa: ¿qué pasa si se separa al azar?

Solo como comparación, se repite el entrenamiento separando las filas al azar en lugar de por IP.

In [ ]:
X_e_azar, X_p_azar, y_e_azar, y_p_azar = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
arbol_azar = crear_arbol(mejor_profundidad).fit(X_e_azar, y_e_azar)

pd.DataFrame({
    "accuracy": [accuracy_score(y_prueba, pred_arbol), accuracy_score(y_p_azar, arbol_azar.predict(X_p_azar))],
}, index=["Separando por IP (honesto)", "Separando al azar (con fuga)"]).round(3)

**Interpretación:** separando al azar, el árbol parece acertar el **69.0 %**, 3.3 puntos más. Esa diferencia es **falsa**:
viene de que el modelo vuelve a ver en la prueba copias de los mismos ataques que memorizó al entrenar.
El resultado honesto, y el que se reporta, es el **65.7 %**, porque mide lo que pasaría con atacantes nuevos.

**Para un analista:** un modelo de seguridad evaluado con fuga de datos falla en producción justo cuando aparece un atacante nuevo.

### 5.2 Regresión lineal: ¿los ataques van en aumento?

Aquí sí hay un número que predecir: **cuántos ataques hay en un día**. Una regresión lineal traza la recta
que mejor sigue los datos a lo largo del tiempo. Su **pendiente** dice cuántos ataques más (o menos) hay cada día.

Hay dos ajustes que salen del análisis anterior:

- Se quitan las **5 IPs de las ráfagas** (sección 4.6). La regresión lineal es muy sensible a los valores extremos:
  un solo día de 13 mil ataques inclinaría toda la recta.
- Se quitan el **primer y el último día**, que están incompletos.

Para evaluarla se entrena con el **80 % inicial** de los días y se prueba con el **20 % final**.
Así se mide si la recta sirve para predecir el futuro. Se compara contra un modelo base que predice siempre el promedio.

In [ ]:
serie = df[~df["srcstr"].isin(top5_ips)].groupby("fecha").size().iloc[1:-1]
dias = np.arange(len(serie)).reshape(-1, 1)  # 0 = primer día completo, 1 = el siguiente, ...

corte = int(len(serie) * 0.8)
regresion = LinearRegression().fit(dias[:corte], serie.values[:corte])
pred_reg = regresion.predict(dias[corte:])
pred_promedio = np.full(len(pred_reg), serie.values[:corte].mean())

print(f"Pendiente: {regresion.coef_[0]:+.2f} ataques por día, cada día")
print(f"Punto de partida (día 0): {regresion.intercept_:,.0f} ataques")
pd.DataFrame({
    "error absoluto medio (ataques/día)": [mean_absolute_error(serie.values[corte:], pred_promedio),
                                           mean_absolute_error(serie.values[corte:], pred_reg)],
    "R²": [r2_score(serie.values[corte:], pred_promedio), r2_score(serie.values[corte:], pred_reg)],
}, index=["Base (promedio)", "Regresión lineal"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fechas = pd.to_datetime(serie.index)
ax.plot(fechas, serie.values, color="lightgray", label="Ataques reales por día")
ax.plot(fechas[:corte], regresion.predict(dias[:corte]), color="steelblue", linewidth=2.5, label="Recta (entrenamiento)")
ax.plot(fechas[corte:], pred_reg, color="red", linewidth=2.5, linestyle="--", label="Recta (predicción del 20 % final)")
ax.axvline(fechas[corte], color="black", linestyle=":", label="Corte entrenamiento / prueba")
ax.set_title("Tendencia de ataques por día (sin las 5 IPs de las ráfagas)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Ataques por día")
ax.legend()
plt.show()

**Interpretación:**

- **Los ataques van en aumento.** La pendiente es de **+2.72 ataques por día, cada día**. La recta arranca en unos 1,883
  ataques diarios en marzo, y al cabo de unos 186 días suma unos **500 ataques diarios más** (cerca de +27 %).
- **Como predictor de un día concreto, la recta es débil.** En el 20 % final se equivoca en promedio por **219 ataques al día**,
  frente a 236 del modelo base (un 7.5 % menos de error). Su R² es de −0.07: la recta no explica las subidas y bajadas
  de cada día, que dependen de campañas puntuales como los picos de la gráfica.
- El modelo base tiene un R² aún peor (−0.60) porque predice el promedio del pasado, y el futuro tuvo **más** ataques.
  Justamente eso es lo que la recta sí captura: la **tendencia**.

**Para un analista:** la regresión sirve para **planear** ("dentro de 6 meses habrá ~25 % más ataques; hay que dimensionar
el monitoreo"), no para adivinar el día exacto del próximo pico.

### 5.3 Visualización final

Un resumen en cuatro gráficas que responde la pregunta del proyecto. Es la misma figura que se usa en la presentación.

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("¿Desde dónde, cuándo y contra qué atacan? — 451,581 ataques a honeypots de AWS (2013)", fontsize=16)

# 1. Desde dónde
paises5 = df["country"].value_counts().head(5).sort_values()
ejes[0, 0].barh(paises5.index, paises5.values / len(df) * 100, color="steelblue")
ejes[0, 0].set_title("1. ¿Desde dónde? China y EE. UU. originan el 62 %")
ejes[0, 0].set_xlabel("% de todos los ataques")

# 2. Cuándo: con y sin las ráfagas
ejes[0, 1].plot(ataques_por_hora.index, ataques_por_hora.values, color="lightgray", marker="o", label="Todos los ataques")
ejes[0, 1].plot(sin_top5.index, sin_top5.values, color="steelblue", marker="o", label="Sin las 5 IPs de ráfagas")
ejes[0, 1].set_title("2. ¿Cuándo? A cualquier hora: no hay horario humano")
ejes[0, 1].set_xlabel("Hora del registro (0–23)")
ejes[0, 1].set_ylabel("Número de ataques")
ejes[0, 1].set_ylim(0)
ejes[0, 1].legend()

# 3. Contra qué
servicios5 = df["servicio"].value_counts().head(5).sort_values()
ejes[1, 0].barh(servicios5.index, servicios5.values / len(df) * 100, color="darkorange")
ejes[1, 0].set_title("3. ¿Contra qué? Bases de datos y Windows")
ejes[1, 0].set_xlabel("% de todos los ataques")

# 4. Quién: curva de concentración (Pareto)
ips_ordenadas = ataques_por_ip.sort_values(ascending=False)
pct_ips = np.arange(1, len(ips_ordenadas) + 1) / len(ips_ordenadas) * 100
pct_ataques = ips_ordenadas.cumsum().values / ips_ordenadas.sum() * 100
ejes[1, 1].plot(pct_ips, pct_ataques, color="seagreen", linewidth=2.5)
ejes[1, 1].axvline(10, color="red", linestyle="--")
ejes[1, 1].annotate(f"El 10 % de las IPs hace el {pct_ataques[int(len(pct_ips) * 0.1) - 1]:.0f} % de los ataques",
                    xy=(10, pct_ataques[int(len(pct_ips) * 0.1) - 1]), xytext=(25, 45),
                    arrowprops={"arrowstyle": "->", "color": "black"}, fontsize=11)
ejes[1, 1].set_title("4. ¿Quién? Pocos atacantes hacen casi todo")
ejes[1, 1].set_xlabel("% de IPs atacantes (de la más activa a la menos)")
ejes[1, 1].set_ylabel("% acumulado de ataques")

plt.tight_layout()
plt.show()

### 5.4 Storytelling: la historia de los datos

**Acto 1 — La trampa.** En marzo de 2013 se encendieron 9 servidores falsos en AWS, sin nada de valor y sin anunciarlos a nadie.
En seis meses recibieron **451,581 ataques**: más de 2,200 al día. Nadie los buscaba a ellos en particular:
**cualquier servidor conectado a internet es atacado, aunque nadie sepa que existe.**

**Acto 2 — Los atacantes.** El 62 % de los ataques salió de máquinas en China y Estados Unidos. Pero el dato más revelador
es otro: **el 10 % de las IPs hizo el 66 % de los ataques**. Casi todos prueban una o dos veces y se van.
Unos pocos insisten miles de veces, como la IP de Irán que lanzó 11,116 ataques en una sola hora.

**Acto 3 — Sin horario y sin descanso.** Al quitar a los atacantes de ráfaga, se ataca casi igual a las 3 a. m. que a las 3 p. m.
No son personas: son **programas automáticos** que trabajan 24/7. Y el problema crece: unos **500 ataques diarios más**
en septiembre que en marzo.

**Acto 4 — Lo que buscan, y si se puede anticipar.** El blanco favorito son las **bases de datos (SQL Server)** y los
**equipos Windows (SMB, RDP)**. Un árbol de decisión logra predecir el servicio atacado en **2 de cada 3 casos**
(65.7 % frente al 45.2 % del modelo base). La pista clave no es el país ni la hora, sino la **huella técnica**
de la herramienta: el puerto de origen 6000.

**Moraleja:** el enemigo no es un hacker de película. Es un ejército de programas automáticos que nunca duermen y buscan siempre
las mismas puertas abiertas. Defenderse empieza por **cerrar esas puertas** y por **vigilar a los pocos que insisten**.

### 5.5 Aplicación profesional

Así usaría este análisis como **analista de un Centro de Operaciones de Seguridad (SOC)**:

| Hallazgo | Acción concreta en el trabajo |
|---|---|
| SQL Server, SMB y RDP son los más atacados | **Endurecimiento (*hardening*):** revisar que ningún servidor de la empresa exponga los puertos 1433, 445 o 3389 a internet; si hace falta acceso remoto, usar una VPN. |
| Una IP normal hace 1 o 2 intentos; el límite de outlier es 8.5 | **Regla de alerta:** generar una alerta en el SIEM cuando una IP supere 9 intentos de conexión, y, por ejemplo, bloquearla automáticamente si supera los 100. |
| El puerto de origen 6000 delata a escáneres de bases de datos | **Firma de detección:** marcar como sospechoso todo tráfico con puerto de origen 6000 hacia puertos de bases de datos. |
| Los ataques no tienen horario | **Turnos y automatización:** el monitoreo debe ser 24/7, idealmente con respuestas automáticas, porque el atacante también es automático. |
| La tendencia sube ~2.7 ataques/día cada día | **Planificación de capacidad:** usar la regresión para estimar cuánto crecerá el volumen de alertas y dimensionar el equipo y las herramientas. |
| El árbol predice el servicio en 2 de cada 3 casos | **Priorización (*triage*):** con los primeros paquetes de una IP nueva, anticipar qué servicio va a atacar y reforzar primero ese servicio. |

Este mismo enfoque (recolectar, limpiar, describir, modelar y comunicar) sirve para cualquier registro de seguridad:
logs de firewall, intentos de inicio de sesión o alertas de antivirus.

## 6. Conclusiones

**Respuesta a la pregunta del proyecto:**

1. **¿Desde dónde?** Sobre todo desde máquinas en **China (42.4 %)** y **Estados Unidos (19.9 %)**, de un total de 177 países.
   Que una IP esté en un país no significa que el atacante viva allí.
2. **¿Cuándo?** **A toda hora.** Sin las ráfagas, la diferencia entre horas es de apenas ~7.5 %: son ataques automáticos.
   Además, **van en aumento**: unos 500 ataques diarios más al final del periodo que al inicio.
3. **¿Contra qué servicios?** **SQL Server** (casi 1 de cada 4 ataques), seguido de los pings de reconocimiento (ICMP)
   y de los servicios de Windows (**SMB** y **RDP**).
4. **¿Se puede predecir el servicio?** **En parte, sí.** El árbol de decisión acierta el **65.7 %**, frente al 45.2 % del modelo base.
   Reconoce muy bien SQL Server y SMB, pero no distingue RDP, que comparte perfil con SQL Server.

**Lo aprendido en el proceso:**

- Los datos reales vienen sucios: una coma corrió 83 filas y el código de Namibia ("NA") se leía como vacío.
- No todo outlier es un error: aquí los valores extremos **eran** los ataques más importantes.
- Con datos repetidos, separar al azar infla los resultados. Separar por IP da un resultado honesto.

**Limitaciones:**

- Los datos son de **2013**: las herramientas y los servicios atacados han cambiado desde entonces.
- No se documenta la **zona horaria**, y la **geolocalización** de una IP no siempre es exacta.
- Un honeypot solo ve ataques **automáticos y masivos**, no ataques dirigidos contra una empresa en particular.
- El modelo usa pocas variables. Con el historial de cada IP o el contenido de los paquetes podría mejorar.